Import required libraries

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

Load API key

In [2]:
load_dotenv()  # Load environment variables from .env file
api_key = os.getenv("OPENAI_API_KEY")  # Get the API key from environment variable

Initiate Open AI instance

In [3]:
client = OpenAI(api_key=api_key)  # Initialize the OpenAI client with the API key

First call test

In [4]:
response = client.chat.completions.create(
    model="gpt-4o",
    temperature=0.2,
    max_tokens=1024,
    messages=[
        {
            "role": "system", 
            "content": "You are a helpful assistant that can answer my simple questions. Your answer should be concise and to the point."
            },
        {
            "role": "user",
            "content": "What places I can visit in New York City if I only have 24 hours to spend there?"
            }
    ]
)

Display response

In [5]:
print(response.choices[0].message.content)

If you have 24 hours in New York City, consider visiting these iconic spots:

1. **Central Park** - Take a stroll or rent a bike to explore this vast urban park.
2. **Times Square** - Experience the vibrant lights and energy of this famous intersection.
3. **Statue of Liberty & Ellis Island** - Take a ferry to see these iconic landmarks.
4. **9/11 Memorial & Museum** - Pay your respects and learn about the events of September 11, 2001.
5. **Empire State Building or Top of the Rock** - Enjoy panoramic views of the city.
6. **Brooklyn Bridge** - Walk across this historic bridge for stunning views of Manhattan.
7. **Fifth Avenue** - Explore high-end shops and landmarks like St. Patrick's Cathedral.
8. **Broadway** - Catch a show if time permits.
9. **The Metropolitan Museum of Art** - Visit one of the world's largest and finest art museums.
10. **Chinatown or Little Italy** - Enjoy diverse culinary experiences.

Plan your route efficiently to make the most of your time.


Test different temperature values

In [6]:
def get_responses(client: OpenAI, temperature: float) -> str | None:
    response = client.chat.completions.create(
        model = "gpt-4o",
        temperature = temperature,
        max_tokens = 1024,
        messages = [
            {
                "role" : "system",
                "content" : "You are a helpful assistant that can answer my simple questions. Your answers should be concise and to the point."
            },
            {
                "role" : "user",
                "content": "What places I can visit in Jersey City if I only have 24 hours to spend there?"
            }
        ]
    )

    return response.choices[0].message.content

In [7]:
temperatures = [0.2, 0.5, 0.8]
for temp in temperatures:
    print("temperature: ", temp)
    print(get_responses(client, temp))
    print("\n")


temperature:  0.2
If you have 24 hours in Jersey City, consider visiting the following places:

1. **Liberty State Park**: Offers stunning views of the Manhattan skyline, the Statue of Liberty, and Ellis Island. Great for a walk or a picnic.

2. **Liberty Science Center**: An interactive science museum with engaging exhibits and an IMAX theater.

3. **Downtown Jersey City**: Explore the vibrant area with its diverse dining options, shops, and street art.

4. **Exchange Place**: A waterfront area with beautiful views of New York City, perfect for a stroll.

5. **Colgate Clock**: A historic landmark and one of the largest clocks in the world, located on the waterfront.

6. **Newport Centre Mall**: If you're interested in shopping, this mall offers a variety of stores and dining options.

7. **Mana Contemporary**: A large contemporary art center with galleries and exhibitions.

8. **Historic Downtown**: Walk around the historic neighborhoods like Paulus Hook and Hamilton Park for charming

Define a chat with memory / chat history

In [8]:
class ChatWithHistory():
    def __init__(self, api_key: str, model: str = "gpt-4o", temp: float = 0.2, max_tokens: int = 1024) -> None:
        self.client = OpenAI(api_key = api_key)
        self.model = model
        self.temp = temp
        self.max_tokens = max_tokens
        self.history = []

    def clear(self) -> None:
        self.history = []
        print("Chat history cleared.")
    
    # Add return value and error handling
    def chat(self, user_input: str) -> str:

        if not self.history:
            self.history.append(
                {
                    "role" : "system",
                    "content" : "You are a helpful assistant that can answer simply questions. Your answers should be concise and to the point."
                }
            )

        print("User: ", user_input)

        self.history.append(
            {
                "role" : "user",
                "content" : user_input
            }
        )

        try:
            message_return = self.client.chat.completions.create(
                model=self.model,
                temperature=self.temp,
                max_tokens=self.max_tokens,
                messages=self.history
            )
        except Exception as e:
            print(f"Error: {e}")
            return ""
        
        response: str = message_return.choices[0].message.content

        print("Assistant: ", response)

        self.history.append(
            {
                "role": "assistant",
                "content": response
            }
        )

        return response  # Return the response


Run the chat

In [9]:
chatbot = ChatWithHistory(api_key)

while True:
    user_input = input("Enter whatever you want: ")

    quit_commands = {"quit", "exit", "bye", "stop"}
    if user_input.lower() in quit_commands:
        break

    if user_input.lower() == "clear":
        chatbot.clear()
    else:
        chatbot.chat(user_input)


User:  How are you?
Assistant:  I'm just a computer program, so I don't have feelings, but I'm here to help you! How can I assist you today?
User:  what is my name?
Assistant:  I'm sorry, I don't have access to personal information, so I don't know your name.
User:  I am Alston.
Assistant:  Nice to meet you, Alston! How can I assist you today?
User:  what is your name?
Assistant:  I’m an AI assistant and don’t have a personal name, but you can call me Assistant. How can I help you today?
User:  I want to call you Mike. Is it OK?
Assistant:  Sure, you can call me Mike if you’d like! How can I assist you today?
User:  What is your name now?
Assistant:  You can call me Mike, as you suggested. How can I help you today?
User:  What is my name then?
Assistant:  You mentioned that your name is Alston.
Chat history cleared.
User:  What is your name now?
Assistant:  I am called Assistant.
User:  Do you still remember my name?
Assistant:  I'm sorry, but I don't have the ability to remember perso